# Run analysis

Learning curves and evaluation results for one training run. Reads only from `results/<run>/`:
the TensorBoard event file, the Monitor CSVs (one row per training episode) and the `eval_*.csv` files written by `evaluate.py`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

RUN = Path("../results/ppo_seed0_20261101_184947")

# Success criteria (same as envs/landing_aviary.py)
MAX_TOUCHDOWN_SPEED = 0.5
MAX_TOUCHDOWN_OFFSET = 0.10

In [ ]:
# Chart style: light surface, recessive grid and axes, thin lines
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "text.color": "#0b0b0b", "axes.labelcolor": "#52514e", "axes.titlecolor": "#0b0b0b",
    "xtick.color": "#898781", "ytick.color": "#898781", "axes.edgecolor": "#c3c2b7",
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "lines.linewidth": 2, "legend.frameon": False, "figure.dpi": 110,
})

# Fixed colour per outcome, so an outcome keeps its colour in every chart
OUTCOME_COLORS = {
    "landed": "#2a78d6",
    "bad_landing": "#eb6834",
    "crashed": "#1baf7a",
    "out_of_bounds": "#eda100",
    "timeout": "#e87ba4",
}
THRESHOLD_STYLE = {"color": "#52514e", "linestyle": "--", "linewidth": 1}

## Learning curves (TensorBoard scalars)

One row per PPO iteration (16,384 steps). Training uses a stochastic policy, so these are noisier than the evaluation below.

In [ ]:
def load_scalars(run: Path) -> pd.DataFrame:
    events = EventAccumulator(str(next(run.glob("tb_*"))), size_guidance={"scalars": 0})
    events.Reload()
    return pd.DataFrame({tag: pd.Series({e.step: e.value for e in events.Scalars(tag)})
                         for tag in events.Tags()["scalars"]})

scalars = load_scalars(RUN)
scalars.index = scalars.index // 1000  # thousands of timesteps, easier to read on the x-axis
scalars.index.name = "timesteps (k)"
scalars[[c for c in scalars if c.startswith("outcome/")]].iloc[::6].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for outcome, color in OUTCOME_COLORS.items():
    ax.plot(scalars.index, scalars[f"outcome/{outcome}"], color=color, label=outcome.replace("_", " "))
ax.axhline(0.9, **THRESHOLD_STYLE)
ax.text(scalars.index[0], 0.92, "milestone: 90% landed", color="#52514e", fontsize=9)
ax.set(title="Episode outcomes during training", xlabel="timesteps (k)", ylabel="share of episodes", ylim=(0, 1.05))
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
fig.tight_layout()
plt.show()

In [ ]:
fig, (ax_return, ax_len) = plt.subplots(1, 2, figsize=(10, 3.5))
ax_return.plot(scalars.index, scalars["rollout/ep_rew_mean"], color="#2a78d6")
ax_return.set(title="Mean episode return", xlabel="timesteps (k)")
ax_len.plot(scalars.index, scalars["rollout/ep_len_mean"] / 30, color="#2a78d6")
ax_len.set(title="Mean episode length (s)", xlabel="timesteps (k)")
fig.tight_layout()
plt.show()

Reward split into its terms. Shows *why* the return changes: first the crash penalty disappears, then the landing bonus takes over.

In [ ]:
terms = [c for c in scalars if c.startswith("episode_reward/")]
fig, axes = plt.subplots(1, len(terms), figsize=(13, 2.8), sharex=True)
for ax, term in zip(axes, terms):
    ax.plot(scalars.index, scalars[term], color="#2a78d6")
    ax.set_title(term.split("/")[1], fontsize=10)
    ax.tick_params(axis="x", labelrotation=30)
fig.suptitle("Reward terms, summed per episode")
fig.tight_layout()
plt.show()

In [ ]:
fig, (ax_speed, ax_offset) = plt.subplots(1, 2, figsize=(10, 3.5))
ax_speed.plot(scalars.index, scalars["touchdown/speed"], color="#2a78d6", marker="o", markersize=3)
ax_speed.axhline(MAX_TOUCHDOWN_SPEED, **THRESHOLD_STYLE)
ax_speed.set(title="Touchdown speed (m/s)", xlabel="timesteps (k)", ylim=(0, None))
ax_offset.plot(scalars.index, scalars["touchdown/offset"], color="#2a78d6", marker="o", markersize=3)
ax_offset.axhline(MAX_TOUCHDOWN_OFFSET, **THRESHOLD_STYLE)
ax_offset.set(title="Touchdown offset from pad centre (m)", xlabel="timesteps (k)", ylim=(0, None))
fig.tight_layout()
plt.show()

## Evaluation (`evaluate.py`)

Fixed seeds, so these numbers are comparable between runs and with the PID baseline.

In [ ]:
evals = {path.stem: pd.read_csv(path) for path in sorted(RUN.glob("eval_*.csv"))}

summary = pd.DataFrame({
    name: {
        "episodes": len(df),
        "success rate": df["success"].mean(),
        "median touchdown speed": df["touchdown_speed"].median(),
        "max touchdown speed": df["touchdown_speed"].max(),
        "median offset": df["touchdown_offset"].median(),
        "max offset": df["touchdown_offset"].max(),
        "median duration (s)": df["duration_s"].median(),
    }
    for name, df in evals.items()
}).T
summary.round(3)

In [ ]:
pd.DataFrame({name: df["outcome"].value_counts() for name, df in evals.items()}).fillna(0).astype(int)

In [ ]:
fig, axes = plt.subplots(1, len(evals), figsize=(5 * len(evals), 4), sharex=True, sharey=True, squeeze=False)
for ax, (name, df) in zip(axes[0], evals.items()):
    ax.add_patch(plt.Rectangle((0, 0), MAX_TOUCHDOWN_OFFSET, MAX_TOUCHDOWN_SPEED,
                               facecolor="#2a78d6", alpha=0.07, edgecolor="none"))
    for outcome in ("landed", "bad_landing"):
        hits = df[df["outcome"] == outcome]
        ax.scatter(hits["touchdown_offset"], hits["touchdown_speed"], s=28,
                   color=OUTCOME_COLORS[outcome], edgecolor="#fcfcfb", linewidth=1,
                   label=f"{outcome.replace('_', ' ')} ({len(hits)})")
    ax.set(title=name, xlabel="touchdown offset (m)")
    ax.legend(loc="upper right")
axes[0][0].set_ylabel("touchdown speed (m/s)")
fig.suptitle("Touchdowns per evaluation episode (shaded = success region)")
fig.tight_layout()
plt.show()